# LangChain Architecture Overview: Chains, Runnables, LCEL, and the New vs Old API

> This notebook contains all code examples from the blog post.
> [Read the full post on BotMartz](https://blog.botmartz.com/langchain_1)

**Author:** Soham Sharma · blog.botmartz.com

In [ ]:
!pip install -q langchain langchain-openai langchain-community chromadb

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Each of these is a Runnable
prompt = ChatPromptTemplate.from_template("Explain {concept} in one sentence.")
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
parser = StrOutputParser()

# The pipe operator | composes Runnables left-to-right
chain = prompt | model | parser

result = chain.invoke({"concept": "attention mechanism"})
print(result)

In [ ]:
from langchain_core.runnables import RunnablePassthrough

passthrough = RunnablePassthrough()
result = passthrough.invoke({"key": "value"})
print(result)

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
parser = StrOutputParser()

pros_chain = (
    ChatPromptTemplate.from_template("List 2 pros of {technology} in bullet points.")
    | model | parser
)
cons_chain = (
    ChatPromptTemplate.from_template("List 2 cons of {technology} in bullet points.")
    | model | parser
)

parallel = RunnableParallel(
    pros=pros_chain,
    cons=cons_chain,
    original=RunnablePassthrough(),
)

result = parallel.invoke({"technology": "vector databases"})
print("PROS:", result["pros"])
print("CONS:", result["cons"])
print("INPUT:", result["original"])

In [ ]:
from langchain_core.runnables import RunnableLambda

def word_count(text: str) -> dict:
    return {"text": text, "word_count": len(text.split())}

counter = RunnableLambda(word_count)
print(counter.invoke("The attention mechanism is powerful"))

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate

system_template = SystemMessagePromptTemplate.from_template(
    "You are an expert in {domain}. Be concise and technical."
)
human_template = HumanMessagePromptTemplate.from_template(
    "Explain: {question}"
)

prompt = ChatPromptTemplate.from_messages([system_template, human_template])

# Format to inspect the messages
messages = prompt.format_messages(domain="distributed systems", question="CAP theorem")
for msg in messages:
    print(f"[{msg.type}] {msg.content}")

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel

class TechSummary(BaseModel):
    name: str
    year_created: int
    primary_use_case: str

model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
parser = JsonOutputParser(pydantic_object=TechSummary)

prompt = ChatPromptTemplate.from_messages([
    ("system", "Return a JSON object matching this schema: {format_instructions}"),
    ("human", "Summarize: {technology}"),
]).partial(format_instructions=parser.get_format_instructions())

chain = prompt | model | parser

result = chain.invoke({"technology": "PyTorch"})
print(result)
print(type(result))

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

chain = (
    ChatPromptTemplate.from_template("What is {topic}?")
    | ChatOpenAI(model="gpt-4o-mini", temperature=0)
    | StrOutputParser()
)

# Correct
result = chain.invoke({"topic": "gradient descent"})

# Wrong — raises ValueError
try:
    chain.invoke("gradient descent")
except Exception as e:
    print(f"Error: {type(e).__name__}: {e}")

print(result[:80])

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

chain = (
    ChatPromptTemplate.from_template("Explain {concept} in 3 sentences.")
    | ChatOpenAI(model="gpt-4o-mini", temperature=0, streaming=True)
    | StrOutputParser()
)

# Stream token by token
for chunk in chain.stream({"concept": "transformer architecture"}):
    print(chunk, end="", flush=True)
print()

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

chain = (
    ChatPromptTemplate.from_template("Summarize {text} in one sentence.")
    | ChatOpenAI(model="gpt-4o-mini")
    | StrOutputParser()
)

print("Input schema:", chain.input_schema.schema())
print("Output schema:", chain.output_schema.schema())